# 06 · Intermediate · Build the policy in code

Every `policy.yaml` setting as Python objects, with the same validation. This is useful
when policies are generated, or for quick what-if runs.

In [ ]:
%pip install /Volumes/main/default/lakehouse_gc/lakehouse_gc-0.4.0-py3-none-any.whl

In [ ]:
from dataclasses import replace
from datetime import date

from lakehouse_gc import run
from lakehouse_gc.core import Policy, load_policy
from lakehouse_gc.core.policy import ActionsConfig, Exclusion, Scope, ScoringConfig

policy = Policy(
    scope=Scope(
        include_catalogs=("main",),  # empty = all; globs like "prod_*" work
        exclude_catalogs=("system", "samples"),
        include_schemas=(),  # catalog.schema
        exclude_schemas=(),
        include_tables=(),  # catalog.schema.table
        exclude_tables=("main.sales.tmp_*",),
    ),
    exclusions=(
        Exclusion("table", "*.__materialization*", "Internal tables"),  # permanent
        Exclusion("table", "main.sales.legacy_*", "Migration hold", date(2026, 12, 31)),
    ),
    sources={"information_schema": "system.information_schema"},  # or "per_catalog"
    rules={
        "uc": {"stale_table_days": 90, "min_table_size_gb": 1, "required_tags": ["cost_center"]}
    },
    scoring=ScoringConfig(
        governance_weight=0.6,  # normalized against cost_weight
        cost_weight=0.4,
        unknown_penalty=0.5,  # cost of a check with unavailable evidence, in (0, 1]
        grade_boundaries=(("A", 90), ("B", 75), ("C", 50), ("D", 0)),  # must include 0
    ),
    actions=ActionsConfig(min_candidate_confidence=80),  # always report-only
)
result = run(policy=policy, scanners=["uc"], spark=spark)
print(result.summary())

In [ ]:
# What-if: start from your policy file and change one setting.
POLICY_PATH = "/Volumes/main/default/lakehouse_gc/policy.yaml"
base = load_policy(POLICY_PATH)
uc_rules = dict(base.rules.get("uc", {}))
stricter = replace(base, rules={**base.rules, "uc": {**uc_rules, "stale_table_days": 30}})

before = run(policy=base, scanners=["uc"], spark=spark)
after = run(policy=stricter, scanners=["uc"], spark=spark)
days = uc_rules.get("stale_table_days", 180)
print(
    f"cost score at {days} days: {before.scores.cost.score}; at 30 days: {after.scores.cost.score}"
)